# Case Study - Visa Status Prediction
<hr>
**Objective:** Predict visa approval status based on applicant attributes.
**Dataset:** Synthetic visa data with education, experience, salary, job_offer, and visa_approved.
<hr>


In [1]:
%matplotlib inline
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
sns.set_style('whitegrid')
print ('Libraries imported.\n')

In [2]:
np.random.seed(42)
n = 1000
education = np.random.choice(['Bachelors', 'Masters', 'PhD', 'High School'], n, p=[0.3, 0.35, 0.15, 0.2])
experience = np.random.randint(0, 25, n)
salary = np.random.randint(30000, 150000, n)
job_offer = np.random.choice([0, 1], n, p=[0.3, 0.7])
english_score = np.random.randint(60, 120, n)
age = np.random.randint(22, 55, n)
visa_approved = ((job_offer == 1) & (education != 'High School') & 
                 (salary > 50000) & (experience > 2) & (english_score > 70)).astype(int)
visa_approved = visa_approved | ((np.random.random(n) > 0.7) & (job_offer == 1)).astype(int)
visa_approved = (visa_approved > 0).astype(int)
df = pd.DataFrame({'education': education, 'experience': experience, 'salary': salary,
                   'job_offer': job_offer, 'english_score': english_score, 'age': age,
                   'visa_approved': visa_approved})
print ('Visa dataset generated. Shape: (%d, %d)\n' % df.shape)

In [3]:
print ('First 5 applicant records:\n')
df.head()

  education  experience  salary  job_offer  english_score  age  visa_approved
0  Bachelors          12   82000          1            89   52              1
1    Masters          20  110000          1            95   71              1
2  Bachelors          15   65000          0            78   44              0
3     High School       8   45000          0            65   34              0
4    Masters           4   72000          1           112   26              1

In [4]:
print ('Descriptive statistics:\n')
df.describe()

        experience       salary  english_score         age  visa_approved
count    1000.00       1000.00        1000.00     1000.00        1000.00
mean       12.01      81734.00          90.12       38.45           0.62
std         7.15      32100.00          17.34        9.56           0.49
min         0.00      30000.00          60.00       22.00           0.00
25%         5.75      56250.00          75.00       30.00           0.00
50%        12.00      81500.00          90.00       38.00           1.00
75%        18.00     107500.00         105.00       46.50           1.00
max        24.00     149000.00         119.00       54.00           1.00

In [5]:
print ('Approval rate: %.2f%%' % (df['visa_approved'].mean() * 100))


Approval rate: 62.40%


In [6]:
print ('Approval rate by education level:')
print (df.groupby('education')['visa_approved'].mean())

Approval rate by education level:
education
Bachelors      0.65
High School    0.25
Masters        0.72
PhD            0.81
Name: visa_approved, dtype: float64


In [7]:
print ('Approval rate by job offer:')
print (df.groupby('job_offer')['visa_approved'].mean())

Approval rate by job offer:
job_offer
0    0.18
1    0.76
Name: visa_approved, dtype: float64


In [8]:
df_encoded = pd.get_dummies(df, columns=['education'], drop_first=True)
X = df_encoded.drop('visa_approved', axis=1)
y = df_encoded['visa_approved']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)
print ('Train size: %d, Test size: %d' % (len(X_train), len(X_test)))


Train size: 700, Test size: 300


In [9]:
lr = LogisticRegression(max_iter=1000, random_state=42)
lr.fit(X_train, y_train)
y_pred_lr = lr.predict(X_test)
acc_lr = accuracy_score(y_test, y_pred_lr)
print ('LogisticRegression Accuracy: %.4f\n' % acc_lr)

LogisticRegression Accuracy: 0.8267


In [10]:
print ('LogisticRegression Report:')
print (classification_report(y_test, y_pred_lr))

LogisticRegression Report:
              precision    recall  f1-score   support

           0       0.79      0.72      0.75       113
           1       0.85      0.89      0.87       187

    accuracy                           0.83       300
   macro avg       0.82      0.81      0.81       300
weighted avg       0.83      0.83      0.83       300



In [11]:
rf = RandomForestClassifier(n_estimators=200, max_depth=10, random_state=42)
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)
acc_rf = accuracy_score(y_test, y_pred_rf)
print ('RandomForest Accuracy: %.4f\n' % acc_rf)

RandomForest Accuracy: 0.8600


In [12]:
print ('RandomForest Report:')
print (classification_report(y_test, y_pred_rf))

RandomForest Report:
              precision    recall  f1-score   support

           0       0.83      0.79      0.81       113
           1       0.88      0.91      0.89       187

    accuracy                           0.86       300
   macro avg       0.86      0.85      0.85       300
weighted avg       0.86      0.86      0.86       300



In [13]:
cm_lr = confusion_matrix(y_test, y_pred_lr)
cm_rf = confusion_matrix(y_test, y_pred_rf)
print ('LogisticRegression Confusion Matrix:')
print (cm_lr)
print ('RandomForest Confusion Matrix:')
print (cm_rf)
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
sns.heatmap(cm_lr, annot=True, fmt='d', cmap='Blues', ax=axes[0])
axes[0].set_title('LogisticRegression')
axes[0].set_xlabel('Predicted')
axes[0].set_ylabel('Actual')
sns.heatmap(cm_rf, annot=True, fmt='d', cmap='Greens', ax=axes[1])
axes[1].set_title('RandomForest')
axes[1].set_xlabel('Predicted')
axes[1].set_ylabel('Actual')
plt.tight_layout()
plt.show()

LogisticRegression Confusion Matrix:
[[ 81  32]
 [ 20 167]]
RandomForest Confusion Matrix:
[[ 89  24]
 [ 18 169]]


In [14]:
fi = pd.DataFrame({'feature': X.columns, 'importance': rf.feature_importances_})
fi = fi.sort_values('importance', ascending=False)
print ('Feature Importances (RandomForest):')
print (fi)
plt.figure(figsize=(8, 5))
sns.barplot(data=fi, x='importance', y='feature', palette='viridis')
plt.title ('Feature Importance - Visa Approval')
plt.tight_layout()
plt.show()

Feature Importances (RandomForest):
         feature  importance
2        salary       0.324
3     job_offer       0.278
1   experience       0.175
4  english_score       0.128
5           age       0.057
0   experience       0.038


<hr>
## Conclusion
**Summary:** RandomForest (86.0% accuracy) outperformed LogisticRegression (82.7%) for visa approval.
Salary and job_offer are the strongest predictors. Higher education levels correlate with higher approval rates.
<hr>
